In [1]:
import dimcli
import pandas as pd
import duckdb
import json

In [2]:
dimcli.login()
dsl = dimcli.Dsl()

Searching config file credentials for default 'live' instance..


Dimcli - Dimensions API Client (v1.7)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.15
Method: dsl.ini file


In [3]:
training_data = pd.read_csv("patents_curated.csv")

In [4]:
training_data

,Rank,publication_number,application_number,family_id,title,abstract,granted_date,granted_year,publication_date,publication_year,...,CSO Categories,IPCR,cpc,id,scope,pillar,research_category,endproduct,ingredient,subpillar
0,7,EP-2079319-B2,EP07816204.7,38969498.0,CONSUMABLES,NaN,2025-05-07,2025.0,2025-05-07,2025,...,NaN,A23L2/60; A23L27/00; A23L27/30; A23L33/105; A2...,A23L29/30; A23L21/00; A23L2/60; A23L27/33; A23...,EP-2079319-B2,out,NaN,NaN,NaN,NaN,NaN
1,16,ES-2644220-T5,ES10751643,42829610.0,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,NaN,2025-04-21,2025.0,2025-04-21,2025,...,NaN,A23C9/12; C12N9/12; C12R1/46; A23L29/269,C12Y207/01006; A23C9/12; A23L33/135; A23V2002/...,ES-2644220-T5,in,PB,Strain development,Yoghurt and fermented dairy,NaN,NaN
2,8,ES-3004341-T3,ES19173302,44510082.0,NUTRITIONAL COMPOSITION,Non-medical use of at least two components sel...,2025-03-12,2025.0,2025-03-12,2025,...,NaN,A61K31/661; A23L33/13; A61K45/06; A61P25/28; A...,A61K31/7068; A61K31/683; A23L33/13; A61K31/498...,ES-3004341-T3,out,NaN,NaN,NaN,NaN,NaN
3,10,EP-3542807-B1,EP19170847.8,44899093.0,"LACTOBACILLUS PENTOSUS LPS01 DSM 21980, L. RHA...",NaN,2025-07-16,2025.0,2025-07-16,2025,...,NaN,A61P1/00; A61K35/745; A23L33/135; A61K35/744; ...,A61K45/06; A61K35/741; Y02A50/30; A61K35/747; ...,EP-3542807-B1,out,NaN,NaN,NaN,NaN,NaN
4,60,US-20250073316-A1,US18954426,45922733.0,FUNCTIONAL FOODS COMPRISING DIAMINE OXIDASE AN...,The present invention relates to functional fo...,NaN,NaN,2025-03-06,2025,...,NaN,A61K9/16; A23C9/12; A61K9/00; A23L33/115; A23L...,A61P21/02; A23C9/1213; A61P43/00; A23C9/13; A2...,US-20250073316-A1,out,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2324,8,BR-112021000359-B1,BR112021000359-2,NaN,FUNGICIDES TO PREVENT AND CONTROL FUNGAL PATHO...,FUNGICIDES FOR PREVENTING AND CONTROLLING FUNG...,NaN,NaN,2025-01-14,2025,...,NaN,A01N47/40; A01P3/00; C07C317/02; A01N47/46; A6...,NaN,BR-112021000359-B1,out,NaN,NaN,NaN,NaN,NaN
2325,20,BR-112021005712-B1,BR112021005712-9,NaN,HEAT TREATED COMPOSITION AND ITS METHOD OF PRE...,"HEAT TREATED COMPOSITION, ITS USE AND ITS METH...",NaN,NaN,2025-01-14,2025,...,NaN,A23L27/21; A23L7/20,NaN,BR-112021005712-B1,out,NaN,NaN,NaN,NaN,NaN
2326,78,HK-40087149-B,HK62023076099A,NaN,METHOD FOR PRODUCING A FOOD PRODUCT,NaN,NaN,NaN,2025-01-10,2025,...,NaN,A23J3/14; A23J3/16; A23J3/22; A23J3/18; A23J3/20,NaN,HK-40087149-B,out,NaN,NaN,NaN,NaN,NaN
2327,178,CA-3293284-A1,CA3293284A,NaN,METHOD FOR OBTAINING A DAIRY ALTERNATIVE FOOD ...,NaN,NaN,NaN,2025-01-09,2025,...,NaN,A23J3/34; A23L2/38; A23C11/10,NaN,CA-3293284-A1,out,NaN,NaN,NaN,NaN,NaN


In [5]:
ids = training_data['id'].tolist()
family_ids = training_data['family_id'].dropna().unique().astype(int).tolist()

In [6]:
len(family_ids)

1434

In [83]:
def chunks(list, n):
    for i in range(0, len(list), n):
        yield list[i:i + n]

In [33]:
for i in range(0, len(ids), 500): print(i)

0
500
1000
1500
2000


In [23]:
## look for all patents with the respective ids
query = []
for i in range(0, len(ids), 500):
    ids_batch = ids[i:i+500]
    q = dsl.query(f"""search patents
          where id in {json.dumps(ids_batch)}
          return patents[id+family_id+application_number+title+abstract+cpc+jurisdiction+current_assignees+current_assignee_names+
                        publication_date+publication_year+priority_year+filing_date+filing_status+original_assignees]
          limit 500""")
    query.append(q)
    

Returned Patents: 500 (total = 500)
Time: 2.68s
Returned Patents: 500 (total = 500)
Time: 2.37s
Returned Patents: 500 (total = 500)
Time: 7.79s
Returned Patents: 500 (total = 500)
Time: 6.22s
Returned Patents: 329 (total = 329)
Time: 2.15s


In [24]:
## look for all patents with the respective family ids
for i in range(0, len(family_ids), 500):
    ids_batch = family_ids[i:i+500]
    q = dsl.query_iterative(f"""search patents
          where family_id in {json.dumps(ids_batch)}
          return patents[id+family_id+application_number+title+abstract+cpc+jurisdiction+
                        publication_date+publication_year+priority_year+filing_date+filing_status+original_assignees] 
          """)
    query.append(q)

Starting iteration with limit=1000 skip=0 ...
0-1000 / 5710 (4.60s)
1000-2000 / 5710 (5.37s)
2000-3000 / 5710 (5.33s)
3000-4000 / 5710 (5.10s)
4000-5000 / 5710 (6.62s)
5000-5710 / 5710 (4.60s)
===
Records extracted: 5710
Starting iteration with limit=1000 skip=0 ...
0-1000 / 2076 (2.42s)
1000-2000 / 2076 (2.75s)
2000-2076 / 2076 (1.89s)
===
Records extracted: 2076
Starting iteration with limit=1000 skip=0 ...
0-958 / 958 (2.49s)
===
Records extracted: 958


In [25]:
# join all query results together
df = pd.concat([q.as_dataframe() for q in query], ignore_index=True)

In [26]:
df

,id,title,abstract,application_number,cpc,family_id,filing_date,filing_status,jurisdiction,original_assignees,priority_year,publication_date,publication_year,current_assignee_names,current_assignees
0,ZA-202309278-B,COLLAGEN HYDROGELS USEFUL AS CELL CARRIERS,<p>It discloses a collagen hydrogel which comp...,ZA2023/09278,"[A61L27/52, A61L27/24, C12N5/0068, A23L13/00, ...",75625524.0,2023-10-04,N/A,ZA,"[{'city_name': 'Cáseda', 'country_code': 'ES',...",2021.0,2025-03-26,2025,NaN,NaN
1,ZA-202307189-B,NUTRIENT MEDIA FOR CELL CULTURE CONTAINING PLA...,<p>The present invention relates to nutritiona...,ZA2023/07189,"[C12N5/0043, A23J3/14, A23J1/14, A23J3/32, A23...",74595223.0,2023-07-18,N/A,ZA,"[{'city_name': 'Uzwil', 'country_code': 'CH', ...",2021.0,2025-10-29,2025,NaN,NaN
2,US-20250386791-A1,SOYBEAN CULTIVAR,"<p id=""p-0001"" num=""0000"">The present inventio...",US19311226,"[A01H5/10, A01H6/542]",72140092.0,2025-08-27,Application,US,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2019.0,2025-12-25,2025,[Syngenta Crop Protection AG Switzerland],"[{'city_name': 'Basel', 'country_code': 'CH', ..."
3,US-20250368377-A1,THIN WALL CONTAINER MADE WITH A RECYCLED MATERIAL,"<p id=""p-0001"" num=""0000"">A thin wall containe...",US19300672,"[B29C2049/023, B65D1/0223, B65D1/0207, B65D1/4...",66334319.0,2025-08-15,Application,US,"[{'city_name': '74500 Evian-Les-Bains', 'count...",2019.0,2025-12-04,2025,[Societe des Eaux Minerales dEvian SA SAEME],"[{'city_name': '74500 Evian-Les-Bains', 'count..."
4,US-20250359561-A1,Automated Food/Feed Mass Transport System,"<p id=""p-0001"" num=""0000"">A food processing li...",US19293250,"[A22C18/00, A22C5/00, A22C7/00, A22C17/0026, A...",67847635.0,2025-08-07,Application,US,"[{'city_name': 'En Bakel', 'country_code': 'NL...",2019.0,2025-11-27,2025,[GEA Food Solutions Bakel BV],"[{'city_name': 'En Bakel', 'country_code': 'NL..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11068,AR-132873-A1,Thin donut-shaped dry kibble for small adult dogs,<p>Methods of feeding small adult dogs include...,ARP240101434,"[A23K50/40, A23K40/00, A23K50/42]",91070315.0,2024-06-05,N/A,AR,"[{'city_name': 'Vevey', 'country_code': 'CH', ...",2023.0,2025-08-06,2025,NaN,NaN
11069,AR-132872-A1,Large donut-shaped dry kibble for small adult ...,<p>Methods of feeding small adult dogs include...,ARP240101433,"[A23K50/40, A23K40/00, A23K50/42]",90922688.0,2024-06-05,N/A,AR,"[{'city_name': 'Vevey', 'country_code': 'CH', ...",2023.0,2025-08-06,2025,NaN,NaN
11070,AR-132615-A1,METHODS AND COMPOSITION FOR APPETIZING PET FOODS,<p>Pet food compositions and methods for makin...,ARP240101149,"[A23K20/147, A23K20/158, A23K20/121, A23K20/13...",90922758.0,2024-05-06,N/A,AR,"[{'city_name': 'Vevey', 'country_code': 'CH', ...",2023.0,2025-07-16,2025,NaN,NaN
11071,AR-131994-A1,Spray-dried milk or milk-based powder composit...,<p>The present disclosure generally relates to...,ARP240100483,"[A23C9/18, C12Y302/01023, A23C1/04, A23C9/1206...",90057519.0,2024-02-28,N/A,AR,"[{'country_code': 'IL', 'country_name': 'Israe...",2023.0,2025-05-21,2025,NaN,NaN


In [27]:
# deduplicate by id
df = df.drop_duplicates(subset="id").reset_index(drop=True)

In [28]:
# remove <..> in abstracts
df['abstract'] = df['abstract'].str.replace(r'<[^>]*>', '', regex=True)

In [ ]:
# check how many NAs in abstracts
any(df["abstract"].isna())
df[df['abstract'].isna()]

,id,title,abstract,application_number,cpc,filing_date,filing_status,jurisdiction,original_assignees,priority_year,publication_date,publication_year
119,RS-67291-B1,COLLAGEN HYDROGELS USEFUL AS CELL CARRIERS,NaN,RS20250969,"[A61L27/24, A23L13/00, C12N2533/54, C12N5/0068...",2022-04-19,N/A,RS,"[{'city_name': 'Cáseda', 'country_code': 'ES',...",2021.0,2025-11-28,2025
120,RS-67239-B1,VEGETARIAN BURGER,NaN,RS20250952,"[A23J3/16, A23J3/14, A23J3/18, A23J3/24, A23L5...",2020-10-20,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2019.0,2025-10-31,2025
121,RS-66799-B1,MEAT ANALOGUE COMPRISING AQUEOUS GELLING COMPO...,NaN,RS20250478,"[A23J3/227, A23V2002/00, A23J3/18, A23J3/16, A...",2018-03-08,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2017.0,2025-06-30,2025
122,RS-66499-B1,MEAT ANALOGUE AND PROCESS FOR PRODUCING THE SAME,NaN,RS20250146,"[A23J3/14, A23L29/035, A23J3/227, A23J3/16]",2021-12-29,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2020.0,2025-03-31,2025
145,JP-7791811-B2,Composition for producing test soils for evalu...,NaN,JP2022501220,"[A61B90/98, B08B2209/08, A61B2090/702, B08B9/4...",2020-07-08,Grant,JP,"[{'city_name': 'Offenburg', 'country_code': 'D...",2019.0,2025-12-24,2025
...,...,...,...,...,...,...,...,...,...,...,...,...
8806,CA-3233687-A1,SOYBEAN VARIETY,NaN,CA3233687,"[A01H6/542, A01H5/10]",2024-03-28,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-10-30,2025
8807,CA-3233203-A1,SOYBEAN VARIETY,NaN,CA3233203,"[A01H6/542, A01H5/10]",2024-03-25,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-07-08,2025
8808,CA-3232136-A1,SOYBEAN VARIETY,NaN,CA3232136,"[A01H6/542, A01H5/10]",2024-03-15,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-07-08,2025
8813,AU-2024395542-A1,SULFONAMIDE DERIVATIVES AND USE THEREOF IN THE...,NaN,AU2024395542,"[C07D413/14, C07D401/14, C07D211/22, C07D401/1...",2024-12-06,Application,AU,"[{'city_name': 'Tours', 'country_code': 'FR', ...",2023.0,2026-05-28,2026


In [39]:
# function to select 1-2 patent per family
def select_patents(group):
    
    preferred_jurisdictions = ['WO', 'EP', 'US']  # note: WO not WP?
    
    # Split into has content / no content
    has_content = group[
        group['title'].notna() & group['abstract'].notna() &
        (group['title'] != '') & (group['abstract'] != '')
    ].copy()
    
    # If nothing has title+abstract, just return newest
    if has_content.empty:
        return group.sort_values('filing_date', ascending=True).head(1)
    
    # Sort by preferred jurisdiction, then newest date
    has_content['_preferred'] = has_content['jurisdiction'].isin(preferred_jurisdictions)
    has_content = has_content.sort_values(
        ['_preferred', 'filing_date'], ascending=[False, True]
    )
    
    # Deduplicate by unique content — keeps best (preferred jurisdiction, newest) per unique text
    unique_content = has_content.drop_duplicates(subset=['title', 'abstract'])
    
    return unique_content.head(2).drop(columns='_preferred')

In [40]:
# get selected patents (reset index part needed to keep family_id)
patents_for_training = df.groupby('family_id', group_keys=True).apply(select_patents).reset_index(level=0).reset_index(drop=True)

In [42]:
patents_for_training['cpc'].dtype

dtype('O')

In [43]:
# join with scope and pillar/category/etc information from training data

# Step 1: filter
mask = (patents_for_training['id'].isin(training_data['id']) | 
        patents_for_training['family_id'].isin(training_data['family_id']))
result = patents_for_training[mask].copy()

# Step 2a: merge by exact id
result = result.merge(training_data[['id', 'scope', 'pillar', 'research_category', 'endproduct', 'ingredient', 'subpillar']], 
                      on='id', how='left')

# Step 2b: merge by family_id as fallback
family_lookup = training_data.drop_duplicates('family_id')[['family_id', 'scope', 'pillar', 'research_category', 'endproduct','ingredient', 'subpillar']]
result = result.merge(family_lookup, on='family_id', how='left', suffixes=('', '_family'))

# Fill gaps from id-merge with family-merge results
result['scope'] = result['scope'].fillna(result['scope_family'])
result['pillar'] = result['pillar'].fillna(result['pillar_family'])
result['research_category'] = result['research_category'].fillna(result['research_category_family'])
result['endproduct'] = result['endproduct'].fillna(result['endproduct_family'])
result['ingredient'] = result['ingredient'].fillna(result['ingredient_family'])
result['subpillar'] = result['subpillar'].fillna(result['subpillar_family'])
result = result.drop(columns=['scope_family', 'pillar_family', 'research_category_family', 'endproduct_family', 'ingredient_family', 'subpillar_family'])

In [45]:
result['cpc'].dtype


dtype('O')

In [185]:
result.to_csv('patents_training_data.csv')

In [10]:
result = pd.read_csv('patents_training_data.csv')

In [7]:
duckdb.connect("../patents_training.db")

### Retrieve the CPC codes that cover all entries for a first filter after the dimensions query

In [46]:
# Get codes that cover all entries with a minimum number of codes
df = result[result['scope'] == 'in']
rows = df['cpc'].dropna().tolist()
uncovered = set(range(len(rows)))
selected_codes = []

while uncovered:
    # Find the code that covers the most uncovered rows
    code_coverage = {}
    for idx in uncovered:
        for code in rows[idx]:
            code_coverage[code] = code_coverage.get(code, set()) | {idx}
    
    best_code = max(code_coverage, key=lambda c: len(code_coverage[c]))
    selected_codes.append(best_code)
    uncovered -= code_coverage[best_code]

print(f"Cover size: {len(selected_codes)}")
print(selected_codes)

Cover size: 108
['A23J3/227', 'A23J3/14', 'A23V2002/00', 'A23C11/10', 'A23L13/00', 'A23J3/20', 'A23C11/103', 'C12N1/205', 'A23C20/02', 'A23L33/185', 'A23C11/02', 'C12N5/0658', 'A23L29/256', 'A47J31/4489', 'A23L27/88', 'C12M25/14', 'A23P30/20', 'C12N9/80', 'C12R2001/645', 'A47J31/4485', 'C07K14/415', 'C12M29/04', 'C12N2510/00', 'A23L25/40', 'C12N9/6483', 'A23G1/48', 'A61Q13/00', 'A23K10/30', 'A23L19/01', 'A23L11/30', 'C12N2501/115', 'C12P19/04', 'C12M41/48', 'A21D2/266', 'B26D2210/02', 'A23J3/16', 'A23L29/10', 'A23C20/00', 'C12N1/14', 'A23G9/42', 'C12N5/0075', 'C12N2513/00', 'A23L27/24', 'A23L33/21', 'A23L2/39', 'C12F3/06', 'C12M23/26', 'A23D9/02', 'C12M25/10', 'C07K14/503', 'A23C11/00', 'A23L11/05', 'G01N33/04', 'C12M29/10', 'C12M23/58', 'A23L33/14', 'C09B61/00', 'C12N9/88', 'C12N1/16', 'A23C9/1526', 'A47J31/401', 'C11C3/10', 'C12N15/67', 'A23C11/06', 'A23J3/10', 'A22C18/00', 'A23J3/26', 'A22C7/0023', 'A61L24/102', 'A23L17/60', 'A23L29/212', 'C07K14/49', 'A61K2800/10', 'A23L11/50', 'B0

In [49]:
# check whether all entries have been covered
covered = df['cpc'].dropna().apply(lambda codes: any(c in selected_codes for c in codes))
assert covered.all()

In [50]:
# save codes
with open('../CPC_for_filter.txt', 'w') as f:
    f.write('\n'.join(selected_codes))

# Read it back in as list
# with open('selected_codes.txt', 'r') as f:
#    selected_codes = f.read().splitlines()

### CPC codes for initial search

In [2]:
cpc_codes = [
    "A23C11/065", "A23C11/10", "A23C11/103", "A23C11/106",
    "A23C20/005", "A23C20/02", "A23C20/025",
    "A23J1/005", "A23J1/006", "A23J1/007", "A23J1/008", "A23J1/009",
    "A23J1/12", "A23J1/14", "A23J1/18",
    "A23J3/14", "A23J3/16", "A23J3/18", "A23J3/20", "A23J3/225", "A23J3/227",
    "A23L11/40", "A23L11/45", "A23L11/50", "A23L11/60", "A23L11/65",
    "A23L15/35", "A23L17/35", "A23L31/00", "A23L33/185", "A23L33/195",
    "A23V2200/264",
    "C12N5/0043", "C12N5/0653", "C12N5/0658"
]

In [3]:
# save codes
with open('../CPC_for_query.txt', 'w') as f:
    f.write('\n'.join(cpc_codes))